<a href="https://colab.research.google.com/github/niranjan2399/ML-and-PyTorch/blob/main/text_classifier_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader
from collections import Counter
from torch.utils.data import random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
dataset = load_dataset('imdb')

split = dataset['train'].train_test_split(test_size=0.1, shuffle=True, seed=42)
train_data = split['train']
validation_data = split['test']
test_data = dataset['test']

print(f"Train Data: {len(train_data)}, Val Data: {len(validation_data)}, Test Data: {len(test_data)}")

In [3]:
def Tokenize(text):
  return text.lower().split()

counter = Counter()
for item in train_data:
  counter.update(Tokenize(item['text']))

vocab = {"<unk>": 0}
for word, freq in counter.items():
  if (freq >= 5): vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab)}")

Vocab size: 43179


In [20]:
def get_indices(text):
  return [vocab.get(word, 0) for word in Tokenize(text)]

def collate_batch(batch):
  labels, texts, offsets = [], [], [0]

  for item in batch:
    labels.append(item['label'])
    indices = torch.tensor(get_indices(item['text']), dtype=torch.long)
    texts.append(indices)
    offsets.append(len(indices))

  labels = torch.tensor(labels, dtype=torch.long)
  texts = torch.cat(texts)
  offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)

  return labels, texts, offsets

In [34]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(validation_data, batch_size=32, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, collate_fn=collate_batch)

In [35]:
class TextClassifier(nn.Module):
  def __init__(self, vocab_size, embed_dim, num_classes):
    super().__init__()

    self.embedding = nn.EmbeddingBag(vocab_size, embed_dim)
    self.dropout = nn.Dropout(0.3)
    self.net = nn.Linear(embed_dim, num_classes)

  def forward(self, text, offsets):
    embedding = self.embedding(text, offsets)
    embedding = self.dropout(embedding)
    return self.net(embedding)

In [36]:
model = TextClassifier(vocab_size=len(vocab), embed_dim=128, num_classes=2)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
loss_fn = nn.CrossEntropyLoss()

In [37]:
best_val_loss = float('inf')
patience = 3
patience_counter = 0

for epoch in range(50):
  model.train()
  total, total_loss, correct = 0, 0, 0

  for labels, text, offsets in train_loader:
    predictions = model.forward(text, offsets)
    loss = loss_fn(predictions, labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    total_loss += loss
    correct = (predictions.argmax(dim=1) == labels).sum().item()
    total += labels.size(0)

  model.eval()
  val_correct, val_total, val_loss = 0, 0, 0

  with torch.no_grad():
    for labels, text, offsets in val_loader:
      predictions = model(text, offsets)
      loss = loss_fn(predictions, labels)

      val_loss += loss
      val_correct += (predictions.argmax(dim=1) == labels).sum().item()
      val_total += labels.size(0)

  if val_loss < best_val_loss:
    best_val_loss = val_loss
    torch.save(model.state_dict(), 'best_model.pt')
    patience_counter = 0
  else:
    patience_counter += 1
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

  print(f"Epoch {epoch}: train_loss={total_loss/total:.4f}, val_loss={val_loss/val_total:.4f}")


Epoch 0: train_loss=0.0205, val_loss=0.0192
Epoch 1: train_loss=0.0178, val_loss=0.0165
Epoch 2: train_loss=0.0156, val_loss=0.0147
Epoch 3: train_loss=0.0141, val_loss=0.0136
Epoch 4: train_loss=0.0131, val_loss=0.0129
Epoch 5: train_loss=0.0123, val_loss=0.0124
Epoch 6: train_loss=0.0117, val_loss=0.0116
Epoch 7: train_loss=0.0112, val_loss=0.0112
Epoch 8: train_loss=0.0107, val_loss=0.0111
Epoch 9: train_loss=0.0103, val_loss=0.0106
Epoch 10: train_loss=0.0100, val_loss=0.0105
Epoch 11: train_loss=0.0098, val_loss=0.0103
Epoch 12: train_loss=0.0095, val_loss=0.0102
Epoch 13: train_loss=0.0093, val_loss=0.0101
Epoch 14: train_loss=0.0091, val_loss=0.0101
Epoch 15: train_loss=0.0089, val_loss=0.0100
Epoch 16: train_loss=0.0087, val_loss=0.0098
Epoch 17: train_loss=0.0086, val_loss=0.0098
Epoch 18: train_loss=0.0084, val_loss=0.0096
Epoch 19: train_loss=0.0083, val_loss=0.0099
Epoch 20: train_loss=0.0082, val_loss=0.0098
Epoch 21: train_loss=0.0081, val_loss=0.0095
Epoch 22: train_loss

In [38]:
model.load_state_dict(torch.load('best_model.pt'))

model.eval()
test_total, test_loss, test_correct = 0, 0, 0

with torch.no_grad():
  for labels, text, offsets in test_loader:
    prediction = model(text, offsets)
    loss = loss_fn(prediction, labels)

    test_loss += loss
    test_correct += (prediction.argmax(dim=1) == labels).sum().item()
    test_total += labels.size(0)

print(f"Test Accuracy: {test_correct/test_total:.4f}")

Test Accuracy: 0.8737
